<a href="https://colab.research.google.com/github/JavierMoralesU/Programaci-n-de-sistemas-base-I/blob/main/Mi_Copia_de_PSSB_I_Colab_U3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación de Sistemas de Base I - Expresiones Regulares (ER) y Autómatas Finitos
**Asignatura:** Programación de Sistemas de Base I (8.º Semestre)  
**Unidad III:** Análisis Léxico  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

---


## 1. Objetivos de Aprendizaje
1. Implementar una clase orientada a objetos que construya la matriz de transición y ejecuta la simulación paso a paso de cadenas de entrada.
2.  Implementar una clase orientada a objetos con algoritmos explícitos para el cálculo de la cerradura epsilon, la función move y la simulación paso a paso del algoritmo.
3. Implementar una arquitectura orientada a objetos para construir autómatas finitos usando el Algoritmo de Thompson para obtener el AFN-ɛ a partir de la ER.
4. Implementar el Algoritmo de Construcción de Subconjuntos utilizando programación orientada a objetos para convertir un AFN en un AFD.

---


## 2. Sección Práctica 1: Matriz de Transición
En esta sección analizaremos el algoritmo simulador para reconocimiento de cadenas mediante un AFD y la matriz de transición.


### [CÓDIGO 1.1] AFDeterminista.py
```python
# ==============================================================================
# Tema: Emulador de Autómata Finito Determinista (AFD) y Matriz de Transición
# ==============================================================================

from typing import Set, Dict, Tuple, List, Optional

class AutomataFinitoDeterminista:
    """Clase orientada a objetos para modelar y simular un AFD (5-tupla)."""

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], int],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions  # Corregido: atributo asignado correctamente
        self.s0 = initial_state
        self.F = accepting_states
        
        # Estado de Error implícito
        self.ERROR_STATE = -1

    def transition(self, state: int, symbol: str) -> int:
        """Consulta la función/matriz de transición delta(s, c)."""
        return self.delta.get((state, symbol), self.ERROR_STATE) # Corregido: llamado al atributo delta

    def evaluate_string(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFD carácter a carácter trazando los estados."""
        current_state = self.s0
        trace = [f"Estado Inicial: q{current_state}"]

        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto Sigma.")
                return False, trace

            next_state = self.transition(current_state, char)
            if next_state == self.ERROR_STATE:
                trace.append(f"  [Bloqueo] Sin transición desde q{current_state} con '{char}'.")
                return False, trace

            trace.append(f"  Paso {i+1}: Símbolo '{char}' | Transición q{current_state} -> q{next_state}")
            current_state = next_state

        is_accepted = current_state in self.F
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estado Final: q{current_state} | Resultado: {status}")
        return is_accepted, trace

    def print_transition_matrix(self):
        """Imprime la representación tabular/matriz de transición del AFD."""
        sorted_symbols = sorted(list(self.Sigma))
        sorted_states = sorted(list(self.S))

        print("\n=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===")
        header = f"{'Estado':<8} | " + " | ".join([f"{sym:^5}" for sym in sorted_symbols])
        print(header)
        print("-" * len(header))

        for state in sorted_states:
            prefix = "-> " if state == self.s0 else (" * " if state in self.F else "   ")
            row_str = f"{prefix}q{state:<4} | "
            row_cells = []
            for sym in sorted_symbols:
                dest = self.transition(state, sym)
                cell = f"q{dest}" if dest != self.ERROR_STATE else "--"
                row_cells.append(f"{cell:^5}")
            row_str += " | ".join(row_cells)
            print(row_str)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción de un AFD para reconocer identificadores simples: Letra (L) seguida de Letras (L) o Dígitos (D)
    # Expresión regular equivalente: L (L | D)*
    
    states_set = {0, 1}
    alphabet_set = {'L', 'D'}
    
    # Transiciones:
    # q0 --L--> q1
    # q1 --L--> q1
    # q1 --D--> q1
    transitions_map = {
        (0, 'L'): 1,
        (1, 'L'): 1,
        (1, 'D'): 1
    }
    
    initial = 0
    accepting = {1}

    afd_id = AutomataFinitoDeterminista(states_set, alphabet_set, transitions_map, initial, accepting)

    # Mostrar la matriz de transición
    afd_id.print_transition_matrix()

    # Batería de pruebas
    test_inputs = ["LLD", "L1D", "DLL", "L"]
    
    print("\n=== EVALUACIÓN DE CADENAS DE PRUEBA ===")
    for test in test_inputs:
        # Convertir caracteres para simplificar la prueba (letras -> 'L', dígitos -> 'D')
        mapped_input = "".join(['L' if c.isalpha() else ('D' if c.isdigit() else c) for c in test])
        print(f"\nEvaluating Input Original: '{test}' -> Abstract: '{mapped_input}'")
        accepted, execution_trace = afd_id.evaluate_string(mapped_input)
        for line in execution_trace:
            print(line)
```



---

In [ ]:
# ==============================================================================
# Tema: Emulador de Autómata Finito Determinista (AFD) y Matriz de Transición
# ==============================================================================

from typing import Set, Dict, Tuple, List, Optional

class AutomataFinitoDeterminista:
    """Clase orientada a objetos para modelar y simular un AFD (5-tupla)."""

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], int],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions  # Corregido: atributo asignado correctamente
        self.s0 = initial_state
        self.F = accepting_states

        # Estado de Error implícito
        self.ERROR_STATE = -1

    def transition(self, state: int, symbol: str) -> int: #Devuelve el estado destino
        """Consulta la función/matriz de transición delta(s, c)."""
        return self.delta.get((state, symbol), self.ERROR_STATE) # Corregido: llamado al atributo delta
        #Delta es un mapa de transiciones. Una transición es un diccionario que relaciona una dupla (ESTADO y SIMBOLO) a un ESTADO destino
        #Ejemplo de una transición   (0, 'L'): 1
        #En un Automata Finito Deterministica la transición solo conecta a un estado, pero en un AFN el diccionario vincula la dupla a una posible
        #lista de estados destinos

    def evaluate_string(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFD carácter a carácter trazando los estados."""
        current_state = self.s0
        trace = [f"Estado Inicial: q{current_state}"]

        for i, char in enumerate(input_string):
            if char not in self.Sigma: #Marca error si se ingresan simbolos que no son parte del abecedario del automata
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto Sigma.")
                return False, trace

            next_state = self.transition(current_state, char) #Recupera el estado/nodo al que se deberia mover segun el simbolo ingresado
            if next_state == self.ERROR_STATE: #Detecta el caso de que no haya ninguna transición posible usando el nuevo simbolo analizado
                trace.append(f"  [Bloqueo] Sin transición desde q{current_state} con '{char}'.")
                return False, trace

            trace.append(f"  Paso {i+1}: Símbolo '{char}' | Transición q{current_state} -> q{next_state}")
            #Registra las transiciones hechas conforme se usan los simbolos
            current_state = next_state # Avanza al nuevo estado actual

        is_accepted = current_state in self.F #Marca como cadena/lexema reconocido si cuando se acaban los simbolos a procesar, el puntero del automata
        # termina en un estado de aceptación, si no lo marca como cadena rechazada, la cadena no pertenece al lenguaje del automata y entonces no se identifica
        # como un lexema valido.
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estado Final: q{current_state} | Resultado: {status}")
        return is_accepted, trace

    def print_transition_matrix(self): #Simplemente muestra la matriz de transición, estados de inicio y al estado destino al que corresponden segun el simbolo
        """Imprime la representación tabular/matriz de transición del AFD."""
        sorted_symbols = sorted(list(self.Sigma))
        sorted_states = sorted(list(self.S))

        print("\n=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===")
        header = f"{'Estado':<8} | " + " | ".join([f"{sym:^5}" for sym in sorted_symbols])
        print(header)
        print("-" * len(header))

        for state in sorted_states:
            prefix = "-> " if state == self.s0 else (" * " if state in self.F else "   ")
            row_str = f"{prefix}q{state:<4} | "
            row_cells = []
            for sym in sorted_symbols:
                dest = self.transition(state, sym)
                cell = f"q{dest}" if dest != self.ERROR_STATE else "--"
                row_cells.append(f"{cell:^5}")
            row_str += " | ".join(row_cells)
            print(row_str)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción de un AFD para reconocer identificadores simples: Letra (L) seguida de Letras (L) o Dígitos (D)
    # Expresión regular equivalente: L (L | D)*

    states_set = {0, 1}
    alphabet_set = {'L', 'D'}

    # Transiciones:
    # q0 --L--> q1
    # q1 --L--> q1
    # q1 --D--> q1
    transitions_map = {
        (0, 'L'): 1,
        (1, 'L'): 1,
        (1, 'D'): 1
    }

    initial = 0
    accepting = {1}

    afd_id = AutomataFinitoDeterminista(states_set, alphabet_set, transitions_map, initial, accepting)

    # Mostrar la matriz de transición
    afd_id.print_transition_matrix()

    # Batería de pruebas
    test_inputs = ["LLD", "L1D", "DLL", "L"]

    print("\n=== EVALUACIÓN DE CADENAS DE PRUEBA ===")
    for test in test_inputs: #Test servira para representar a cada una de las cadenas a probar, hay 4
        # Convertir caracteres para simplificar la prueba (letras -> 'L', dígitos -> 'D')
        #Lee caracter por caracter identificando y marcando si es letra, digito o si mejor se deja tal cual, estos caracteres se unen para reformar la cadena
        mapped_input = "".join(['L' if c.isalpha() else ('D' if c.isdigit() else c) for c in test])

        print(f"\nEvaluating Input Original: '{test}' -> Abstract: '{mapped_input}'")
        accepted, execution_trace = afd_id.evaluate_string(mapped_input) #Recupera el procedimiento de transiciones y si la cadena fue aceptada o no
        for line in execution_trace:
            print(line)


=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===
Estado   |   D   |   L  
------------------------
-> q0    |  --   |  q1  
 * q1    |  q1   |  q1  

=== EVALUACIÓN DE CADENAS DE PRUEBA ===

Evaluating Input Original: 'LLD' -> Abstract: 'LLL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'L' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'L1D' -> Abstract: 'LDL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'D' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'DLL' -> Abstract: 'LLL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'L' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'L' -> Abstract: 'L'
Estado Inicial: q0

## 3. Sección Práctica 2: Cerradura-ɛ
En esta sección construiremos una clase orientada a objetos con algoritmos explícitos para el cálculo de la cerradura epsilon y la función move.

### [CÓDIGO 2.1] epsilonClosure.py
```python
# ==============================================================================
# Tema: Emulador de Autómata Finito No Determinista con Transiciones Epsilon (AFN-e)
# ==============================================================================

from typing import Set, Dict, Tuple, List, Any

class NFA_Epsilon:
    """Clase orientada a objetos para modelar y simular un AFN con transiciones epsilon."""
    EPSILON = 'ε'  # Cadena vacía constante

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], Set[int]],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions
        self.s0 = initial_state
        self.F = accepting_states

    def epsilon_closure_state(self, state: int) -> Set[int]:
        """Calcula la epsilon-cerradura para un solo estado usando un algoritmo de pila (DFS)."""
        closure = {state}
        stack = [state]

        while stack:
            current = stack.pop()
            # Buscar transiciones epsilon desde el estado actual
            next_states = self.delta.get((current, self.EPSILON), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def epsilon_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula la epsilon-cerradura para un conjunto de estados: e-closure(T) = U e-closure(s)."""
        total_closure = set()
        for state in states_set:
            total_closure.update(self.epsilon_closure_state(state))
        return total_closure

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula el conjunto mover(T, a): estados alcanzables con el símbolo 'a'."""
        reachable = set()
        for state in states_set:
            nxt = self.delta.get((state, symbol), set())
            reachable.update(nxt)
        return reachable

    def simulate(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFN (Algoritmo 3.22 del Dragon Book) sobre una cadena."""
        trace = []
        # Paso 1: Estado inicial activo S = e-closure(s0)
        current_active = self.epsilon_closure_state(self.s0)
        trace.append(f"Estado Activo Inicial e-closure({self.s0}): {sorted(list(current_active))}")

        # Bucle principal sobre cada carácter de la entrada
        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto.")
                return False, trace

            # Paso 2a: T = mover(S, c)
            move_set = self.move(current_active, char)
            # Paso 2b: S' = e-closure(T)
            next_active = self.epsilon_closure_set(move_set)

            trace.append(f"  Paso {i+1} ('{char}'): mover -> {sorted(list(move_set))} | e-closure -> {sorted(list(next_active))}")
            current_active = next_active

            if not current_active:
                trace.append("  [Bloqueo] El conjunto de estados activos quedó vacío.")
                return False, trace

        # Criterio de Aceptación: S ∩ F != Ø
        is_accepted = bool(current_active.intersection(self.F))
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estados Activos Finales: {sorted(list(current_active))} | Intersección F: {sorted(list(current_active.intersection(self.F)))}")
        trace.append(f"Resultado: {status}")

        return is_accepted, trace

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción del AFN-e para el lenguaje (a|b)*abb
    states_all = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
    alphabet_in = {'a', 'b'}
    
    # Transiciones del AFN-e (basado en la figura 3.34 de Aho et al.)
    transitions_nfa = {
        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }
    
    start_st = 0
    accept_st = {10}

    afn = NFA_Epsilon(states_all, alphabet_in, transitions_nfa, start_st, accept_st)

    print("=== EMULADOR DE AFN-e (Simulación Directa) ===")
    test_string = "aabb"
    print(f"\nEvaluando la cadena: '{test_string}'")
    
    accepted, trace_log = afn.simulate(test_string)
    for line in trace_log:
        print(line)
```

In [ ]:
# ==============================================================================
# Tema: Emulador de Autómata Finito No Determinista con Transiciones Epsilon (AFN-e)
# ==============================================================================

from typing import Set, Dict, Tuple, List, Any

class NFA_Epsilon: #Un NFA tiene que pensar en conjuntos de estados y ramificaciones ya que un mismo simbolo entrada (tambien incluyendo epsilon para este caso) puede llevar a diferentes transiciones.
#Un DFA es deterministico, un solo estado destino y por ende lineal.
    """Clase orientada a objetos para modelar y simular un AFN con transiciones epsilon."""
    EPSILON = 'ε'  # Cadena vacía constante

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], Set[int]], #Ahora las duplas estan vinculadas a una lista de estados destino en lugar de uno solo.
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions
        self.s0 = initial_state
        self.F = accepting_states

    def epsilon_closure_state(self, state: int) -> Set[int]: #Recorre las transiciones vacias mediante el uso de una pila, y guarda en una lista el alcanze expandido del nodo seleccionado
        """Calcula la epsilon-cerradura para un solo estado usando un algoritmo de pila (DFS)."""

        stack = [state] #Lista temporal para recorrer el automata desde el nodo "actual" y navegando a travez de cada una de las ramas de transiciones vacias
        closure = {state} #Lista que se devolvera. Incluira a todos los nodos alcanzables desde el nodo "actual" mediante transiciones vacias (no solo vecinos directos)


        while stack:
            current = stack.pop()
            # Buscar transiciones epsilon desde el estado actual
            next_states = self.delta.get((current, self.EPSILON), set())
            #Recupera la lista de los nodos vecinos directos a las que se puede acceder desde nodo actual usando una unica transición vacia (parece no incluirse a si mismo, pero daria igual para la lista)

            for nxt in next_states: #Se revisa cada uno de estos nodos vecinos por transición vacia
                if nxt not in closure: #Solo se agrega a la lista de nodos alcanzables, y a la pila, si es que es un nodo nuevo
                    closure.add(nxt)
                    stack.append(nxt) #La pila nos permitira revisar uno por uno cada nodo y a sus vecinos, todo con el objetivo de llenar la lista de alcanze expandio por la clausula epsilon
        return closure

    def epsilon_closure_set(self, states_set: Set[int]) -> Set[int]:# Calculo de expansión que trata a los simbolos "vacios" de epsilon
        """Calcula la epsilon-cerradura para un conjunto de estados: e-closure(T) = U e-closure(s)."""
        total_closure = set()
        for state in states_set:
            total_closure.update(self.epsilon_closure_state(state)) #El metodo calculara el alcanze expandido de cada estado volviendolo un conjunto de estados,
            # y cada uno de estos conjuntos se une para formar la lista de clausura total
        return total_closure

    def move(self, states_set: Set[int], symbol: str) -> Set[int]: #Navegación que devuelve los estados destino a los que pueden llegar las diferentes ramificaciones (Usa simbolos de verdad, no le importa epsilon)
        """Calcula el conjunto mover(T, a): estados alcanzables con el símbolo 'a'."""
        reachable = set()
        for state in states_set:
            nxt = self.delta.get((state, symbol), set())
            reachable.update(nxt)
        return reachable

    def simulate(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFN (Algoritmo 3.22 del Dragon Book) sobre una cadena."""
        trace = []
        # Paso 1: Estado inicial activo S = e-closure(s0) | Inicio en el nodo 0 y se calcula su alcanze ampliado con el metodo que maneja las transiciones vacias
        current_active = self.epsilon_closure_state(self.s0)
        trace.append(f"Estado Activo Inicial e-closure({self.s0}): {sorted(list(current_active))}")

        # Bucle principal sobre cada carácter de la entrada
        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto.")
                return False, trace

            # Paso 2a: T = mover(S, c)
            move_set = self.move(current_active, char) #Movimiento real usando simbolos. Es una lista de estados destino
            # Paso 2b: S' = e-closure(T)
            next_active = self.epsilon_closure_set(move_set) #Ampliación que calcula el nuevo alcance avanzando por las transiciones vacias.
            # Es un conjunto que incluye la lista de estados destino, más los estados/nodos alacanzables mediante navegar las transiciones vacias.

            trace.append(f"  Paso {i+1} ('{char}'): mover -> {sorted(list(move_set))} | e-closure -> {sorted(list(next_active))}")
            current_active = next_active

            if not current_active: #Aunque se movio y se expandio, la lista de nodos quedo vacia (Se hayo un estado trampa). En teoria no deberia ser posible si es que se tomaran en cuenta transiciones vacias hacia si mismo.
                trace.append("  [Bloqueo] El conjunto de estados activos quedó vacío.")
                return False, trace

        # Criterio de Aceptación: S ∩ F != Ø
        is_accepted = bool(current_active.intersection(self.F)) #Se marca como cadena aceptada si el conjunto de estados actuales incluye a alguno de los estados de aceptación "self.F"
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estados Activos Finales: {sorted(list(current_active))} | Intersección F: {sorted(list(current_active.intersection(self.F)))}")
        trace.append(f"Resultado: {status}")

        return is_accepted, trace

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción del AFN-e para el lenguaje (a|b)*abb
    states_all = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
    alphabet_in = {'a', 'b'}

    # Transiciones del AFN-e (basado en la figura 3.34 de Aho et al.)
    transitions_nfa = {
        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }

    start_st = 0
    accept_st = {10} #Un unico estado individual de aceptación, pero cualquier conjunto que lo incluya deberia pasar a ser considerado un estado de aceptación

    afn = NFA_Epsilon(states_all, alphabet_in, transitions_nfa, start_st, accept_st)

    print("=== EMULADOR DE AFN-e (Simulación Directa) ===")
    test_string = "aabb"
    print(f"\nEvaluando la cadena: '{test_string}'")

    accepted, trace_log = afn.simulate(test_string)
    for line in trace_log:
        print(line)
    '''
    Evaluando la cadena: 'aabb'
    Estado Activo Inicial e-closure(0): [0, 1, 2, 4, 7]
    Paso 1 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
    Paso 2 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
    Paso 3 ('b'): mover -> [5, 9] | e-closure -> [1, 2, 4, 5, 6, 7, 9]
    Paso 4 ('b'): mover -> [5, 10] | e-closure -> [1, 2, 4, 5, 6, 7, 10]
    Estados Activos Finales: [1, 2, 4, 5, 6, 7, 10] | Intersección F: [10]
    Resultado: CADENA ACEPTADA

        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}

    La ejecución muestra al estado expandido de s0 como [0, 1, 2, 4, 7] debido a que primero el 0 se expande a incluir 1 y 7, y despues el estado 1 tambien se expande a incluir 2 y 4 mediante epsilon.
    Despues se intenta hacer el movimiento con a a travez de cada uno de estos [0, 1, 2, 4, 7], de estos solo existe las transiciones de 2-> {3}  y 7-> {8}. El conjunto de estados destino [3,8] pasa a ser expandido
    y se transforma en [1, 2, 3, 4, 6, 7, 8] ya que  las expansiones por transiciones vacias fueron asi 3->6 , 6-> 1 , 6-> 7, 1->2 , 1->4  y el propio 8 que no se expandio más alla de a si mismo.

    '''

=== EMULADOR DE AFN-e (Simulación Directa) ===

Evaluando la cadena: 'aabb'
Estado Activo Inicial e-closure(0): [0, 1, 2, 4, 7]
  Paso 1 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
  Paso 2 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
  Paso 3 ('b'): mover -> [5, 9] | e-closure -> [1, 2, 4, 5, 6, 7, 9]
  Paso 4 ('b'): mover -> [5, 10] | e-closure -> [1, 2, 4, 5, 6, 7, 10]
Estados Activos Finales: [1, 2, 4, 5, 6, 7, 10] | Intersección F: [10]
Resultado: CADENA ACEPTADA


## 4. Sección Práctica 3: ER a AFN-ɛ
En esta sección implementaremos una arquitectura OO para obtener el AFN-ɛ a partir de la ER.

### [CÓDIGO 3.1] ERAFNepsilon.py
```python
# ==============================================================================
# Tema: Algoritmo de Construcción de Thompson
# ==============================================================================

from typing import Set, Dict, Tuple, List

class State:
    """Representa un estado individual de un autómata Thompson."""
    _id_counter = 0

    def __init__(self):
        self.id = State._id_counter
        State._id_counter += 1
        # Diccionario de transiciones: símbolo -> lista de estados destino
        self.transitions: Dict[str, List['State']] = {}

    def add_transition(self, symbol: str, target_state: 'State'):
        if symbol not in self.transitions:
            self.transitions[symbol] = []
        self.transitions[symbol].append(target_state)

    def __repr__(self):
        return f"q{self.id}"

class ThompsonNFA:
    """Encapsula un sub-autómata Thompson con exactamente UN inicio y UN fin."""
    EPSILON = 'ε'

    def __init__(self, start: State, accept: State):
        self.start = start
        self.accept = accept

    @staticmethod
    def build_symbol(symbol: str) -> 'ThompsonNFA':
        """Regla Base: Construcción para un símbolo a in Sigma (o epsilon)."""
        start = State()
        accept = State()
        start.add_transition(symbol, accept)
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_union(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 1: Unión (nfa1 | nfa2)."""
        start = State()
        accept = State()
        
        # Transiciones epsilon desde nuevo inicio a inicios de nfa1 y nfa2
        start.add_transition(ThompsonNFA.EPSILON, nfa1.start)
        start.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        
        # Transiciones epsilon desde finales de nfa1 y nfa2 a nuevo final
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, accept)
        nfa2.accept.add_transition(ThompsonNFA.EPSILON, accept)
        
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_concat(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 2: Concatenación (nfa1 . nfa2)."""
        # Conectar final de nfa1 con inicio de nfa2 mediante epsilon
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        return ThompsonNFA(nfa1.start, nfa2.accept)

    @staticmethod
    def build_kleene(nfa: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 3: Clausura de Kleene (nfa*)."""
        start = State()
        accept = State()
        
        # 1. Transición epsilon de bypass (0 repeticiones)
        start.add_transition(ThompsonNFA.EPSILON, accept)
        # 2. Transición epsilon hacia inicio del sub-autómata
        start.add_transition(ThompsonNFA.EPSILON, nfa.start)
        # 3. Transición epsilon desde final del sub-autómata hacia nuevo final
        nfa.accept.add_transition(ThompsonNFA.EPSILON, accept)
        # 4. Transición epsilon de retroceso (loop de repetición)
        nfa.accept.add_transition(ThompsonNFA.EPSILON, nfa.start)
        
        return ThompsonNFA(start, accept)

    def print_nfa(self):
        """Imprime la lista de adyacencia de todas las transiciones del autómata."""
        visited = set()
        stack = [self.start]
        print(f"\n=== AFN Thompson (Inicio: {self.start}, Aceptación: {self.accept}) ===")
        
        while stack:
            state = stack.pop()
            if state.id in visited:
                continue
            visited.add(state.id)
            
            for symbol, targets in state.transitions.items():
                for target in targets:
                    print(f"  {state} --({symbol})--> {target}")
                    if target.id not in visited:
                        stack.append(target)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Reset contador de estados
    State._id_counter = 0

    # Construir inductivamente el AFN para la expresión: (a | b)*
    nfa_a = ThompsonNFA.build_symbol('a')      # a
    nfa_b = ThompsonNFA.build_symbol('b')      # b
    nfa_ab = ThompsonNFA.build_union(nfa_a, nfa_b)  # (a | b)
    nfa_star = ThompsonNFA.build_kleene(nfa_ab)     # (a | b)*

    nfa_star.print_nfa()
```


In [ ]:
# ==============================================================================
# Tema: Algoritmo de Construcción de Thompson
# ==============================================================================

from typing import Set, Dict, Tuple, List

class State: #Estructura para representar a cada estado de manera individual, cuenta con un diccionario donde se registran su abecedario y transiciones
    """Representa un estado individual de un autómata Thompson."""
    _id_counter = 0

    def __init__(self):
        self.id = State._id_counter
        State._id_counter += 1
        # Diccionario de transiciones: símbolo -> lista de estados destino
        self.transitions: Dict[str, List['State']] = {} # ejemplo del diccionario:  [ Simbolo 'a' - {lista de estados} ]

    def add_transition(self, symbol: str, target_state: 'State'): # Simbolo y estado destino
        if symbol not in self.transitions: #Si es un nuevo simbolo, pasa a incluirse en el diccionario. Como tal no hay una lista de abecedario global, sino que cada estado tendra sus simbolos registrados
            self.transitions[symbol] = []
        self.transitions[symbol].append(target_state) #Agrega el nuevo estado a la lista de estados vinculada al simbolo.

    def __repr__(self):
        return f"q{self.id}"

class ThompsonNFA:
    """Encapsula un sub-autómata Thompson con exactamente UN inicio y UN fin."""
    EPSILON = 'ε' #Se declara a el simbolo epsilon como el caracter ε

    def __init__(self, start: State, accept: State):
        self.start = start
        self.accept = accept #Solo un estado de aceptación

    @staticmethod
    def build_symbol(symbol: str) -> 'ThompsonNFA': #Genera mini automatas simples de un solo simbolo. Representa a las expresiones simples como "a" = O--a-->O
        """Regla Base: Construcción para un símbolo a in Sigma (o epsilon)."""
        start = State() #Estado de inicio de este automata especifico
        accept = State() #Estado fin de este automata especifico
        start.add_transition(symbol, accept) #Se crea la transición del nodo de inicio al destino usando el simbolo
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_union(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA': #Genera un automata más complejo compuesto de otros 2 automatas,  dos nodos auxiliares y 4 transiciones vacias.
    # Representa la estructura de union '|' que es una bifurcación
        """Regla Inductiva 1: Unión (nfa1 | nfa2)."""
        start = State() #Nuevo nodo auxiliar (inicio)
        accept = State() #Nuevo nodo auxiliar (destino)

        # Transiciones epsilon desde nuevo inicio a inicios de nfa1 y nfa2
        start.add_transition(ThompsonNFA.EPSILON, nfa1.start)
        start.add_transition(ThompsonNFA.EPSILON, nfa2.start)

        # Transiciones epsilon desde finales de nfa1 y nfa2 a nuevo final
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, accept)
        nfa2.accept.add_transition(ThompsonNFA.EPSILON, accept)
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_concat(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA': #Genera un automata un poco más complejo compuesto de otros 2 automatas conectados mediante una transición vacia.
        """Regla Inductiva 2: Concatenación (nfa1 . nfa2)."""
        # Conectar final de nfa1 con inicio de nfa2 mediante epsilon
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        return ThompsonNFA(nfa1.start, nfa2.accept)

    @staticmethod
    def build_kleene(nfa: 'ThompsonNFA') -> 'ThompsonNFA': #Automata de estructura compleja compuesta de 1 automata, 2 nodos auxiliares y 4 transiciones vacias.
    # Es la estructura de clausura de kleene '*' la cual permite el procesamiento de 0 o n veces un simbolo mediante el uso de transiciones para crear bucles
        """Regla Inductiva 3: Clausura de Kleene (nfa*)."""
        start = State()
        accept = State()

        # 1. Transición epsilon de bypass (0 repeticiones)
        start.add_transition(ThompsonNFA.EPSILON, accept)
        # 2. Transición epsilon hacia inicio del sub-autómata
        start.add_transition(ThompsonNFA.EPSILON, nfa.start)
        # 3. Transición epsilon desde final del sub-autómata hacia nuevo final
        nfa.accept.add_transition(ThompsonNFA.EPSILON, accept)
        # 4. Transición epsilon de retroceso (loop de repetición)
        nfa.accept.add_transition(ThompsonNFA.EPSILON, nfa.start)
        return ThompsonNFA(start, accept)

    def print_nfa(self): #Recorre el automata y lo imprime
        """Imprime la lista de adyacencia de todas las transiciones del autómata."""
        visited = set()
        stack = [self.start] #Estructura usada para navegar los distintos caminos del automata
        print(f"\n=== AFN Thompson (Inicio: {self.start}, Aceptación: {self.accept}) ===")

        while stack:
            state = stack.pop()
            if state.id in visited:
                continue #Si es un estado que ya a sido visitado, el ciclo pasa directamente a su siguiente iteración
            visited.add(state.id)

            for symbol, targets in state.transitions.items(): # Las transiciones se guardan en un diccionaario, .items esta devolviendo algo asi: 'ε': [q1, q3]  o 'a': [q2]
            #Los simbolos son una capa superior en el bucle, un simbolo puede conllevar varios estados destino
                for target in targets: #Recorre estado por estado
                    print(f"  {state} --({symbol})--> {target}")
                    if target.id not in visited:
                        stack.append(target) #Se encargara de poder explorar los caminos que salen de los nodos que aun no habian sido visitados

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Reset contador de estados
    State._id_counter = 0

    # Construir inductivamente el AFN para la expresión: (a | b)*
    nfa_a = ThompsonNFA.build_symbol('a')      # a , automata simple
    nfa_b = ThompsonNFA.build_symbol('b')      # b , automata simple
    nfa_ab = ThompsonNFA.build_union(nfa_a, nfa_b)  # (a | b) , automata que une los dos anteriores
    nfa_star = ThompsonNFA.build_kleene(nfa_ab)     # (a | b)* , al anterior se le aplica la clausula de kleene

    nfa_star.print_nfa()


=== AFN Thompson (Inicio: q6, Aceptación: q7) ===
  q6 --(ε)--> q7
  q6 --(ε)--> q4
  q4 --(ε)--> q0
  q4 --(ε)--> q2
  q2 --(b)--> q3
  q3 --(ε)--> q5
  q5 --(ε)--> q7
  q5 --(ε)--> q4
  q0 --(a)--> q1
  q1 --(ε)--> q5


## 5. Sección Práctica 4: ER a AFN-ɛ
En esta sección implementaremos el Algoritmo de Construcción de Subconjuntos para convertir un AFN en un AFD.

### [CÓDIGO 4.1] AFN-AFD.py
```python
# ==============================================================================
# Tema: Algoritmo de Construcción de Subconjuntos
# ==============================================================================

from typing import Set, Dict, Tuple, List, Optional

class SubsetConstructionConverter:
    """Implementación formal de la Construcción de Subconjuntos (Algoritmo 3.20 Aho et al.)."""
    EPSILON = 'ε'

    def __init__(self, nfa_states: Set[int], alphabet: Set[str],
                 nfa_transitions: Dict[Tuple[int, str], Set[int]],
                 nfa_start_state: int, nfa_accept_states: Set[int]):
        self.nfa_states = nfa_states
        self.alphabet = alphabet
        self.nfa_delta = nfa_transitions
        self.nfa_start = nfa_start_state
        self.nfa_accept = nfa_accept_states

    def e_closure_state(self, state: int) -> Set[int]:
        """Calcula e-closure para un estado individual mediante algoritmo de pila."""
        closure = {state}
        stack = [state]
        while stack:
            curr = stack.pop()
            next_states = self.nfa_delta.get((curr, self.EPSILON), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def e_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula e-closure(T) = U e-closure(s) para s in T."""
        total = set()
        for s in states_set:
            total.update(self.e_closure_state(s))
        return total

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula mover(T, a): estados alcanzables con 'a' desde T."""
        reachable = set()
        for s in states_set:
            nxt = self.nfa_delta.get((s, symbol), set())
            reachable.update(nxt)
        return reachable

    def convert(self) -> Tuple[List[Set[int]], Dict[Tuple[int, str], int], Set[int]]:
        """Ejecuta el Algoritmo de Construcción de Subconjuntos."""
        # 1. Estado Inicial del AFD: A = e-closure(s0)
        start_closure = self.e_closure_state(self.nfa_start)
        
        d_states: List[Set[int]] = [start_closure]  # Lista de conjuntos Dstates
        d_tran: Dict[Tuple[int, str], int] = {}     # Matriz Dtran[state_idx, symbol]
        unmarked_indices = [0]                      # Cola de estados no marcados

        while unmarked_indices:
            curr_idx = unmarked_indices.pop(0)
            curr_set = d_states[curr_idx]

            for symbol in sorted(list(self.alphabet)):
                # U = e-closure(mover(T, a))
                move_res = self.move(curr_set, symbol)
                u_set = self.e_closure_set(move_res)

                if not u_set:
                    continue  # Transición vacía / no existe

                # Comprobar si U ya existe en Destados
                if u_set in d_states:
                    target_idx = d_states.index(u_set)
                else:
                    d_states.append(u_set)
                    target_idx = len(d_states) - 1
                    unmarked_indices.append(target_idx)

                d_tran[(curr_idx, symbol)] = target_idx

        # 2. Identificar estados finales del AFD (los que contienen al menos un final del AFN)
        dfa_accept_states = set()
        for idx, d_set in enumerate(d_states):
            if bool(d_set.intersection(self.nfa_accept)):
                dfa_accept_states.add(idx)

        return d_states, d_tran, dfa_accept_states

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # AFN para (a|b)*abb (Ejemplo 3.21 de Aho et al.)
    nfa_S = set(range(11))
    nfa_Sigma = {'a', 'b'}
    nfa_transitions_map = {
        (0, 'ε'): {1, 7},
        (1, 'ε'): {2, 4},
        (2, 'a'): {3},
        (3, 'ε'): {6},
        (4, 'b'): {5},
        (5, 'ε'): {6},
        (6, 'ε'): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }
    nfa_s0 = 0
    nfa_F = {10}

    converter = SubsetConstructionConverter(nfa_S, nfa_Sigma, nfa_transitions_map, nfa_s0, nfa_F)
    dfa_nodes, dfa_matrix, dfa_accept = converter.convert()

    print("=== RESULTADO DE LA CONSTRUCCIÓN DE SUBCONJUNTOS ===")
    print(f"Total de estados AFD generados: {len(dfa_nodes)}\n")

    state_labels = [chr(65 + i) for i in range(len(dfa_nodes))]  # Nombres A, B, C, D, E...

    for idx, node_set in enumerate(dfa_nodes):
        is_acc = " [ACEPTACIÓN]" if idx in dfa_accept else ""
        print(f"Estado AFD {state_labels[idx]} ({idx}): {sorted(list(node_set))}{is_acc}")

    print("\n=== MATRIZ DE TRANSICIÓN Dtran ===")
    print(f"{'Estado Orig':<12} | {'Símbolo':<8} | {'Estado Destino':<12}")
    print("-" * 40)
    for (src_idx, sym), tgt_idx in dfa_matrix.items():
        print(f"{state_labels[src_idx]} ({src_idx}){'':<6} | {sym:<8} | {state_labels[tgt_idx]} ({tgt_idx})")
```

In [ ]:
# ==============================================================================
# Tema: Algoritmo de Construcción de Subconjuntos
# ==============================================================================

from typing import Set, Dict, Tuple, List, Optional

class SubsetConstructionConverter: #Es para convertir automatas no deterministicos, a deterministicos
    """Implementación formal de la Construcción de Subconjuntos (Algoritmo 3.20 Aho et al.)."""
    EPSILON = 'ε'

    def __init__(self, nfa_states: Set[int], alphabet: Set[str],
                 nfa_transitions: Dict[Tuple[int, str], Set[int]],
                 nfa_start_state: int, nfa_accept_states: Set[int]):
        self.nfa_states = nfa_states #Lista de estados
        self.alphabet = alphabet  #Alfabeto de simbolos
        self.nfa_delta = nfa_transitions #Transiciones, vuelve a ser un dicionario con formato:  (estado, simbolo) -> Lista de estados destino
        self.nfa_start = nfa_start_state #Estado inicial
        self.nfa_accept = nfa_accept_states #Lista de estados de aceptación

    def e_closure_state(self, state: int) -> Set[int]: #Realiza el calculo de expansión de alcanze de un estado, devolviendo a todos los nodos alacnzables por transiciones vacias
        """Calcula e-closure para un estado individual mediante algoritmo de pila."""
        closure = {state} #Guardado del subconjunto de estados alcanzables desde el estado actual mediante epsilon
        stack = [state] #Estructura auxiliar para la navegación
        while stack:
            curr = stack.pop()
            next_states = self.nfa_delta.get((curr, self.EPSILON), set()) #Da el estado actual y simbolo epsilon para obtener del dicionario la lista de estados vecinos directos con ε
            for nxt in next_states: #Explora cada uno de los vecinos directos, si aun no estan visitados, estos se agregan a la lista del subconjunto y tambien a la pila para asi
                if nxt not in closure: #Poder seguir explorando a profundidad todos los nodos accesibles mediante ε.
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def e_closure_set(self, states_set: Set[int]) -> Set[int]: #Esta función recibe una lista de estados destino (puede ser una lista de un solo estado), utiliza la función anterior
        """Calcula e-closure(T) = U e-closure(s) para s in T.""" # para aplicarle la expansión a cada de estos estados y junta los subconjuntos resultantes para tener una lista expandida
        total = set()
        for s in states_set:
            total.update(self.e_closure_state(s))
        return total

    def move(self, states_set: Set[int], symbol: str) -> Set[int]: #La operación de movimiento que busca dentro de una lista de estados expandidos, si es posible realizar alguna transición
        """Calcula mover(T, a): estados alcanzables con 'a' desde T.""" # con un simbolo de verdad (no cuenta ε), y devuele una lista de estados destino.
        reachable = set()
        for s in states_set:
            nxt = self.nfa_delta.get((s, symbol), set())
            reachable.update(nxt)
        return reachable

    def convert(self) -> Tuple[List[Set[int]], Dict[Tuple[int, str], int], Set[int]]:
        """Ejecuta el Algoritmo de Construcción de Subconjuntos."""
        # 1. Estado Inicial del AFD: A = e-closure(s0)
        start_closure = self.e_closure_state(self.nfa_start) #Se realiza la expansión de la clausura epsilon al estado inicial.

        #Estas variables registraran los nuevos estados (creados a partir de conjuntos de estados del AFN) para el automata deterministico
        # y las nuevas transiciones de este nuevo automata, siguiendo el mismo tipo de diccionario con formato: (estado, simbolo) -> un unico estados destino
        d_states: List[Set[int]] = [start_closure]  # Lista de conjuntos Dstates
        d_tran: Dict[Tuple[int, str], int] = {}     # Matriz Dtran[state_idx, symbol]
        unmarked_indices = [0]                      # Cola de estados no marcados. Almacena los indices, no los conjuntos como tal. Servira como la estructura de visitados para navegar el AFN

        while unmarked_indices: # Es una cola de estados AFD, osea corresponden a un conjunto de estados AFN expandidos por la clausura epsilon
            curr_idx = unmarked_indices.pop(0) #ejemplo  {0, 1, 2, 4, 7} ->  {1, 2, 4, 7}, curr_idx = 0 y curr_set ={0, 1, 2, 4, 7}
            curr_set = d_states[curr_idx] #Usa el indice/ estado a checar para recuperar el conjunto expandido de estados alcanzables de la lista.
            # lista d_states [indice 0 : {0, 1, 2, 4, 7} ]

            for symbol in sorted(list(self.alphabet)): #Para más orden se ordenan los simbolos para poder mantener una estructura consistente. {a,b}
                # U = e-closure(mover(T, a))
                move_res = self.move(curr_set, symbol) #Obtiene la lista de estados destino alcanzable para el simbolo revisado en la iteración actual.
                u_set = self.e_closure_set(move_res) #Expande cada uno de los estados destino, obteniendo el alcanze completo tomado en cuenta la transiciones con el simbolo y
                # los nodos alcanzados con las transiciones vacias. Ejemplos:  move_res = {3,8} /expansión de clausura epsilon/ u_set = conjunto {1,2,3,4,6,7,8}

                if not u_set: #No se pudo alcanzar ningun nodo ni con el simbolo ni con transiciones vacias, quedo un conjunto vacio
                    continue  # Transición vacía / no existe. En teoria no deberia ser posible siendo que las transiciones vacias tambien apuntan al propio estado en cuestión

                # Comprobar si U ya existe en Destados. Se comprueba si este conjunto de estados AFN coincide con uno ya registrado en los estados del AFD
                # Los estados de AFD estan compuestos de conjuntos de estados de AFN, mientras que de AFN obtienes el conjunto {1,2,3,4,6,7,8}, para el AFD esto puede pasar a ser llamado
                # simplemente estado A
                if u_set in d_states:
                    target_idx = d_states.index(u_set) #Identifica cual era el indice de este conjunto en la lista de estados del AFD
                else:
                    d_states.append(u_set) #Crea el nuevo estado del automata determinista con el conjunto de estados AFN
                    target_idx = len(d_states) - 1 #Le asigna un nuevo nombre / indice a este estado. En este caso se usa numeros, en otros ejercicios estos estados pasaban a usar A, B, C, etc
                    unmarked_indices.append(target_idx) #Añade a la lista de conjuntos por checar

                d_tran[(curr_idx, symbol)] = target_idx #Registra la transición:  (estado AFD actual, simbolo) ->  estado AFD destino. Ejemplo d_tran[(0, 'a')] = 1 ___  d_tran[(0, 'b')] = 2

        # 2. Identificar estados finales del AFD (los que contienen al menos un final del AFN)
        dfa_accept_states = set()
        for idx, d_set in enumerate(d_states): #Navega indice/nombre del estado AFD, e internamente a cada uno de los estados AFN dentro del conjunto
            if bool(d_set.intersection(self.nfa_accept)): #Valida a los conjuntos(estados) del automata determinista que incluyan a alguno de los estados de aceptación del AFN
                dfa_accept_states.add(idx)

        return d_states, d_tran, dfa_accept_states #Devuelve los estados del AFD, sus transiciones, y cuales son sus estados de aceptación

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # AFN para (a|b)*abb (Ejemplo 3.21 de Aho et al.)
    nfa_S = set(range(11)) #Estados
    nfa_Sigma = {'a', 'b'} #Alfabeto de simbolos
    nfa_transitions_map = { #Transiciones del automata no determinista que permite uso de epsilon
        (0, 'ε'): {1, 7},
        (1, 'ε'): {2, 4},
        (2, 'a'): {3},
        (3, 'ε'): {6},
        (4, 'b'): {5},
        (5, 'ε'): {6},
        (6, 'ε'): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }
    nfa_s0 = 0 #estado inicial
    nfa_F = {10} #lista de estados de aceptación, aunque en este caso solo hay uno

    converter = SubsetConstructionConverter(nfa_S, nfa_Sigma, nfa_transitions_map, nfa_s0, nfa_F) #Construcción del automata finito no deterministico
    dfa_nodes, dfa_matrix, dfa_accept = converter.convert() #Conversión a automata finito deterministico, recuperando sus nuevos estados, transiciones y estados de aceptación

    print("=== RESULTADO DE LA CONSTRUCCIÓN DE SUBCONJUNTOS ===")
    print(f"Total de estados AFD generados: {len(dfa_nodes)}\n")

    state_labels = [chr(65 + i) for i in range(len(dfa_nodes))]  # Nombres A, B, C, D, E...

    for idx, node_set in enumerate(dfa_nodes):
        is_acc = " [ACEPTACIÓN]" if idx in dfa_accept else ""
        print(f"Estado AFD {state_labels[idx]} ({idx}): {sorted(list(node_set))}{is_acc}")

    print("\n=== MATRIZ DE TRANSICIÓN Dtran ===")
    print(f"{'Estado Orig':<12} | {'Símbolo':<8} | {'Estado Destino':<12}")
    print("-" * 40)
    for (src_idx, sym), tgt_idx in dfa_matrix.items():
        print(f"{state_labels[src_idx]} ({src_idx}){'':<6} | {sym:<8} | {state_labels[tgt_idx]} ({tgt_idx})")

    '''
     === RESULTADO DE LA CONSTRUCCIÓN DE SUBCONJUNTOS ===
    Total de estados AFD generados: 5
    Estado AFD A (0): [0, 1, 2, 4, 7]
    Estado AFD B (1): [1, 2, 3, 4, 6, 7, 8]
    Estado AFD C (2): [1, 2, 4, 5, 6, 7]
    Estado AFD D (3): [1, 2, 4, 5, 6, 7, 9]
    Estado AFD E (4): [1, 2, 4, 5, 6, 7, 10] [ACEPTACIÓN]
    #Muestra como los estados del nuevo automata deterministico son en realidad conjuntos de estados del automata no deterministico.
    # El estado E es el unico estado de aceptación del AFD debido a ser el unico conjunto que incluye al estado de aceptación del AFN (10)

    === MATRIZ DE TRANSICIÓN Dtran ===
    Estado Orig  | Símbolo  | Estado Destino
    ----------------------------------------
    A (0)       | a        | B (1)
    A (0)       | b        | C (2)
    B (1)       | a        | B (1)
    B (1)       | b        | D (3)
    C (2)       | a        | B (1)
    C (2)       | b        | C (2)
    D (3)       | a        | B (1)
    D (3)       | b        | E (4)
    E (4)       | a        | B (1)
    E (4)       | b        | C (2)
    #Muestra las transiciones entre los estados del AFD. Ya no hay transiciones vacias, y un simbolo en un estado solo puede tener un estado destino
    '''

=== RESULTADO DE LA CONSTRUCCIÓN DE SUBCONJUNTOS ===
Total de estados AFD generados: 5

Estado AFD A (0): [0, 1, 2, 4, 7]
Estado AFD B (1): [1, 2, 3, 4, 6, 7, 8]
Estado AFD C (2): [1, 2, 4, 5, 6, 7]
Estado AFD D (3): [1, 2, 4, 5, 6, 7, 9]
Estado AFD E (4): [1, 2, 4, 5, 6, 7, 10] [ACEPTACIÓN]

=== MATRIZ DE TRANSICIÓN Dtran ===
Estado Orig  | Símbolo  | Estado Destino
----------------------------------------
A (0)       | a        | B (1)
A (0)       | b        | C (2)
B (1)       | a        | B (1)
B (1)       | b        | D (3)
C (2)       | a        | B (1)
C (2)       | b        | C (2)
D (3)       | a        | B (1)
D (3)       | b        | E (4)
E (4)       | a        | B (1)
E (4)       | b        | C (2)


  # **Inicio de desafios ##################################**

## Desafío 1: Exploración y Modificación de un AFD (Nivel: Básico)
Script de referencia: AFDeterminista.py

**Objetivo:** Comprender la estructura de una 5-tupla, el manejo de diccionarios como funciones de transición y la representación tabular.

**Consigna para el estudiante:**

>Ejecución y análisis: Capture y ejecute el programa original. Observe detenidamente cómo se imprime la matriz de transición (`Dtran`) y cómo se evalúan las cadenas `"LLD"`, `"L1D"`, etc.

>Modificación del alfabeto y el lenguaje: Modifique el autómata para que, además de reconocer identificadores simples (`L(L|D)*`), acepte también números enteros positivos (por ejemplo, una secuencia de uno o más dígitos: `D+`).

 ***Se entendio la instrucción como que el automata sea capaz de validar cadenas en plan a(b|5)*: a , ab, a5, ab55b como "identificador simple" y a cadenas como 1, 12, 1223334444, etc como "digitos"***

>Ampliación de pruebas: Agregue al arreglo `test_inputs` al menos tres cadenas adicionales (una válida que comience con dígitos, otra mixta y una inválida) y documente paso a paso mediante comentarios en su código por qué el AFD rechaza o acepta cada una.

In [ ]:
# ==============================================================================
# Tema: Emulador de Autómata Finito Determinista (AFD) y Matriz de Transición
# ==============================================================================

from typing import Set, Dict, Tuple, List, Optional

class AutomataFinitoDeterminista:
    """Clase orientada a objetos para modelar y simular un AFD (5-tupla)."""

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], int],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions  # Corregido: atributo asignado correctamente
        self.s0 = initial_state
        self.F = accepting_states

        # Estado de Error implícito
        self.ERROR_STATE = -1

    def transition(self, state: int, symbol: str) -> int: #Devuelve el estado destino
        """Consulta la función/matriz de transición delta(s, c)."""
        return self.delta.get((state, symbol), self.ERROR_STATE) # Corregido: llamado al atributo delta
        #Delta es un mapa de transiciones. Una transición es un diccionario que relaciona una dupla (ESTADO y SIMBOLO) a un ESTADO destino
        #Ejemplo de una transición   (0, 'L'): 1
        #En un Automata Finito Deterministica la transición solo conecta a un estado, pero en un AFN el diccionario vincula la dupla a una posible
        #lista de estados destinos

    def evaluate_string(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFD carácter a carácter trazando los estados."""
        current_state = self.s0
        trace = [f"Estado Inicial: q{current_state}"]

        for i, char in enumerate(input_string):
            if char not in self.Sigma: #Marca error si se ingresan simbolos que no son parte del abecedario del automata
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto Sigma.")
                return False, trace

            next_state = self.transition(current_state, char) #Recupera el estado/nodo al que se deberia mover segun el simbolo ingresado
            if next_state == self.ERROR_STATE: #Detecta el caso de que no haya ninguna transición posible usando el nuevo simbolo analizado
                trace.append(f"  [Bloqueo] Sin transición desde q{current_state} con '{char}'.")
                return False, trace

            trace.append(f"  Paso {i+1}: Símbolo '{char}' | Transición q{current_state} -> q{next_state}")
            #Registra las transiciones hechas conforme se usan los simbolos
            current_state = next_state # Avanza al nuevo estado actual

        is_accepted = current_state in self.F #Marca como cadena/lexema reconocido si cuando se acaban los simbolos a procesar, el puntero del automata
        # termina en un estado de aceptación, si no lo marca como cadena rechazada, la cadena no pertenece al lenguaje del automata y entonces no se identifica
        # como un lexema valido.
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estado Final: q{current_state} | Resultado: {status}")
        return is_accepted, trace

    def print_transition_matrix(self): #Simplemente muestra la matriz de transición, estados de inicio y al estado destino al que corresponden segun el simbolo
        """Imprime la representación tabular/matriz de transición del AFD."""
        sorted_symbols = sorted(list(self.Sigma))
        sorted_states = sorted(list(self.S))

        print("\n=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===")
        header = f"{'Estado':<8} | " + " | ".join([f"{sym:^5}" for sym in sorted_symbols])
        print(header)
        print("-" * len(header))

        for state in sorted_states:
            prefix = "-> " if state == self.s0 else (" * " if state in self.F else "   ")
            row_str = f"{prefix}q{state:<4} | "
            row_cells = []
            for sym in sorted_symbols:
                dest = self.transition(state, sym)
                cell = f"q{dest}" if dest != self.ERROR_STATE else "--"
                row_cells.append(f"{cell:^5}")
            row_str += " | ".join(row_cells)
            print(row_str)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción de un AFD para reconocer identificadores simples: Letra (L) seguida de Letras (L) o Dígitos (D)
    # Expresión regular equivalente: L (L | D)*

    states_set = {0, 1, 2} #Solo se requieren de 3 estados, 0 que es el nodo que se ramifica a los otros dos. 1 - identificador, 2 - numero
    alphabet_set = {'L', 'D'}

    transitions_map = {
        # Estado inicial
        (0, 'L'): 1,  # Cadena inicia con una letra
        (0, 'D'): 2,  # Cadena inicia con un digito
        # Transiciones de Identificador simple (L(L|D)*). La cadena puede usar letra o digito en cualquier orden y veces que quiera
        (1, 'L'): 1,
        (1, 'D'): 1,
        # Transiciones de numero (D+). Esta rama solo acepta añadirle más digitos
        (2, 'D'): 2
    }

    initial = 0
    accepting = {1,2} #1 corresponde a identificador simple, 2 a numero entero positivo

    afd_id = AutomataFinitoDeterminista(states_set, alphabet_set, transitions_map, initial, accepting)

    # Mostrar la matriz de transición
    afd_id.print_transition_matrix()

    # Batería de pruebas
    test_inputs = ["LLD", "L1D", "DLL", "L", "123", "AB3C", "5a" ] # nuevas pruebas : dígitos, otra mixta y una inválida
    ''' Nuevas cadenas de pruebas ###
        Evaluating Input Original: '123' -> Abstract: 'DDD'
        Estado Inicial: q0
        Paso 1: Símbolo 'D' | Transición q0 -> q2
        Paso 2: Símbolo 'D' | Transición q2 -> q2
        Paso 3: Símbolo 'D' | Transición q2 -> q2
        Estado Final: q2 | Resultado: CADENA ACEPTADA
        La cadena 123 se transforma a DDD, y sigue el siguiente recorrido: 0->2->2. Es una cadena aceptada y que sigue la logica de D+, el nodo 2 permite procesar cuantos simbolos D hayan en la cadena

        Evaluating Input Original: 'AA3' -> Abstract: 'LLD'
        Estado Inicial: q0
        Paso 1: Símbolo 'L' | Transición q0 -> q1
        Paso 2: Símbolo 'L' | Transición q1 -> q1
        Paso 3: Símbolo 'D' | Transición q1 -> q1
        Estado Final: q1 | Resultado: CADENA ACEPTADA
        La cadena AA3 se transforma en LLD, y sigue el recorrido: 0->1->1. Es una cadena aceptada que sigue la logica de (L(L|D)*), el nodo 1 le permite hacer las transiciones L o D que quiera

        Evaluating Input Original: '5a' -> Abstract: 'DL'
        Estado Inicial: q0
        Paso 1: Símbolo 'D' | Transición q0 -> q2
        [Bloqueo] Sin transición desde q2 con 'L'.
        La cadena 5a se transforma en DL, y sigue el recorrido 0->2->bloqueo. No es una cadana aceptada debido a que cuando llega al nodo 2 ya no se permite usar el simbolo L al no existir una transición para esto.
    '''
    print("\n=== EVALUACIÓN DE CADENAS DE PRUEBA ===")
    for test in test_inputs: #Test servira para representar a cada una de las cadenas a probar, hay 4
        # Convertir caracteres para simplificar la prueba (letras -> 'L', dígitos -> 'D')
        #Lee caracter por caracter identificando y marcando si es letra, digito o si mejor se deja tal cual, estos caracteres se unen para reformar la cadena
        mapped_input = "".join(['L' if c.isalpha() else ('D' if c.isdigit() else c) for c in test])

        print(f"\nEvaluating Input Original: '{test}' -> Abstract: '{mapped_input}'")
        accepted, execution_trace = afd_id.evaluate_string(mapped_input) #Recupera el procedimiento de transiciones y si la cadena fue aceptada o no
        for line in execution_trace:
            print(line)


=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===
Estado   |   D   |   L  
------------------------
-> q0    |  q2   |  q1  
 * q1    |  q1   |  q1  
 * q2    |  q2   |  --  

=== EVALUACIÓN DE CADENAS DE PRUEBA ===

Evaluating Input Original: 'LLD' -> Abstract: 'LLL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'L' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'L1D' -> Abstract: 'LDL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'D' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'DLL' -> Abstract: 'LLL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'L' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'L' -> Abstrac

##Desafío 2: Rastreo de Pila y Clausuras Epsilon en AFN (Nivel: Básico-Intermedio)  

Script de referencia: epsilonClosure.py  

**Objetivo:** Analizar el algoritmo basado en pilas (DFS) para calcular la épsilon-clausura y entender la simulación de caminos múltiples no deterministas.  

**Consigna para el estudiante:**  

>Trazado manual vs. computacional: Siga el rastro impreso por el emulador para la cadena `"aabb"`. Identifique visualmente en qué momento exacto el conjunto de estados activos se bifurca gracias a las transiciones $\varepsilon$.  

>Instrumentación del código: Modifique el método `epsilon_closure_state` para que imprima en consola cada vez que un estado entra y sale de la pila (`stack`). Esto le permitirá visualizar el recorrido tipo DFS (Depth-First Search).  

>Prueba de estrés de diseño: Cambie la cadena de prueba a `"aa"` y a `"abbb"`. Analice la traza de ejecución e identifique en qué paso exacto ocurre un `[Bloqueo]` o por qué se rechaza la cadena según el criterio de intersección con el conjunto de aceptación `F`.

In [ ]:
# ==============================================================================
# Tema: Emulador de Autómata Finito No Determinista con Transiciones Epsilon (AFN-e)
# ==============================================================================

from typing import Set, Dict, Tuple, List, Any

class NFA_Epsilon: #Un NFA tiene que pensar en conjuntos de estados y ramificaciones ya que un mismo simbolo entrada (tambien incluyendo epsilon para este caso) puede llevar a diferentes transiciones.
#Un DFA es deterministico, un solo estado destino y por ende lineal.
    """Clase orientada a objetos para modelar y simular un AFN con transiciones epsilon."""
    EPSILON = 'ε'  # Cadena vacía constante

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], Set[int]], #Ahora las duplas estan vinculadas a una lista de estados destino en lugar de uno solo.
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions
        self.s0 = initial_state
        self.F = accepting_states

    def epsilon_closure_state(self, state: int) -> Set[int]: #Recorre las transiciones vacias mediante el uso de una pila, y guarda en una lista el alcanze expandido del nodo seleccionado
        """Calcula la epsilon-cerradura para un solo estado usando un algoritmo de pila (DFS)."""

        stack = [state] #Lista temporal para recorrer el automata desde el nodo "actual" y navegando a travez de cada una de las ramas de transiciones vacias
        closure = {state} #Lista que se devolvera. Incluira a todos los nodos alcanzables desde el nodo "actual" mediante transiciones vacias (no solo vecinos directos)

        print(f"La pila comienza con el estado '{state}'. Pila = {stack}")

        while stack:
            current = stack.pop()
            print(f"El estado '{current}' salio de la pila. Pila = {stack}")


            # Buscar transiciones epsilon desde el estado actual
            next_states = self.delta.get((current, self.EPSILON), set())
            #Recupera la lista de los nodos vecinos directos a las que se puede acceder desde nodo actual usando una unica transición vacia (parece no incluirse a si mismo, pero daria igual para la lista)

            for nxt in next_states: #Se revisa cada uno de estos nodos vecinos por transición vacia
                if nxt not in closure: #Solo se agrega a la lista de nodos alcanzables, y a la pila, si es que es un nodo nuevo
                    closure.add(nxt)
                    stack.append(nxt) #La pila nos permitira revisar uno por uno cada nodo y a sus vecinos, todo con el objetivo de llenar la lista de alcanze expandio por la clausula epsilon
                    print(f"El estado '{nxt}' entro en la pila. Pila = {stack}")
        print(f"La lista de estados visitados con ayuda de la pila es = {closure}\n")

        return closure

    def epsilon_closure_set(self, states_set: Set[int]) -> Set[int]:# Calculo de expansión que trata a los simbolos "vacios" de epsilon
        """Calcula la epsilon-cerradura para un conjunto de estados: e-closure(T) = U e-closure(s)."""
        total_closure = set()
        for state in states_set:
            total_closure.update(self.epsilon_closure_state(state)) #El metodo calculara el alcanze expandido de cada estado volviendolo un conjunto de estados,
            # y cada uno de estos conjuntos se une para formar la lista de clausura total
        return total_closure

    def move(self, states_set: Set[int], symbol: str) -> Set[int]: #Navegación que devuelve los estados destino a los que pueden llegar las diferentes ramificaciones (Usa simbolos de verdad, no le importa epsilon)
        """Calcula el conjunto mover(T, a): estados alcanzables con el símbolo 'a'."""
        reachable = set()
        for state in states_set:
            nxt = self.delta.get((state, symbol), set())

            reachable.update(nxt)
        print(f"En los estados expandidos {states_set} se uso la transición {symbol}-> y se pudo llegar a {reachable}")
        return reachable

    def simulate(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFN (Algoritmo 3.22 del Dragon Book) sobre una cadena."""
        trace = []
        # Paso 1: Estado inicial activo S = e-closure(s0) | Inicio en el nodo 0 y se calcula su alcanze ampliado con el metodo que maneja las transiciones vacias
        current_active = self.epsilon_closure_state(self.s0)
        trace.append(f"Estado Activo Inicial e-closure({self.s0}): {sorted(list(current_active))}")

        # Bucle principal sobre cada carácter de la entrada
        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto.")
                return False, trace

            # Paso 2a: T = mover(S, c)
            move_set = self.move(current_active, char) #Movimiento real usando simbolos. Es una lista de estados destino
            print(f"Lista de estados destino a expandir: {move_set}###############\n")

            # Paso 2b: S' = e-closure(T)
            next_active = self.epsilon_closure_set(move_set) #Ampliación que calcula el nuevo alcance avanzando por las transiciones vacias.
            # Es un conjunto que incluye la lista de estados destino, más los estados/nodos alacanzables mediante navegar las transiciones vacias.

            trace.append(f"  Paso {i+1} ('{char}'): mover -> {sorted(list(move_set))} | e-closure -> {sorted(list(next_active))}")
            current_active = next_active

            if not current_active: #Aunque se movio y se expandio, la lista de nodos quedo vacia (Se hayo un estado trampa). En teoria no deberia ser posible si es que se tomaran en cuenta transiciones vacias hacia si mismo.
                trace.append("  [Bloqueo] El conjunto de estados activos quedó vacío.")
                return False, trace

        # Criterio de Aceptación: S ∩ F != Ø
        is_accepted = bool(current_active.intersection(self.F)) #Se marca como cadena aceptada si el conjunto de estados actuales incluye a alguno de los estados de aceptación "self.F"
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estados Activos Finales: {sorted(list(current_active))} | Intersección F: {sorted(list(current_active.intersection(self.F)))}")
        trace.append(f"Resultado: {status}")

        return is_accepted, trace

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción del AFN-e para el lenguaje (a|b)*abb
    states_all = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
    alphabet_in = {'a', 'b'}

    # Transiciones del AFN-e (basado en la figura 3.34 de Aho et al.)
    transitions_nfa = {
        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }

    start_st = 0
    accept_st = {10} #Un unico estado individual de aceptación, pero cualquier conjunto que lo incluya deberia pasar a ser considerado un estado de aceptación

    afn = NFA_Epsilon(states_all, alphabet_in, transitions_nfa, start_st, accept_st)

    print("=== EMULADOR DE AFN-e (Simulación Directa) ===")

    test_string = "aa"
    print(f"\nEvaluando la cadena: '{test_string}'")
    accepted, trace_log = afn.simulate(test_string)
    for line in trace_log:
        print(line)
    '''
    Estado Activo Inicial e-closure(0): [0, 1, 2, 4, 7]
    Paso 1 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
    Paso 2 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
    Estados Activos Finales: [1, 2, 3, 4, 6, 7, 8] | Intersección F: []
    Resultado: CADENA RECHAZADA
    No llega a ocurrir ningun bloqueo,si se procesan todos los simbolos pero al final el conjunto de estados resultante de hacer la ultima
    transición no incluye al estado 10 del AFN, lo que significa que la cadena 'aa' no termino en un estado de aceptación del AFD y por ende no es una cadena valida.
    '''
    test_string = "abbb"
    print(f"\nEvaluando la cadena: '{test_string}'")
    accepted, trace_log = afn.simulate(test_string)
    for line in trace_log:
        print(line)
    '''
    Estado Activo Inicial e-closure(0): [0, 1, 2, 4, 7]
    Paso 1 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
    Paso 2 ('b'): mover -> [5, 9] | e-closure -> [1, 2, 4, 5, 6, 7, 9]
    Paso 3 ('b'): mover -> [5, 10] | e-closure -> [1, 2, 4, 5, 6, 7, 10]
    Paso 4 ('b'): mover -> [5] | e-closure -> [1, 2, 4, 5, 6, 7]
    Estados Activos Finales: [1, 2, 4, 5, 6, 7] | Intersección F: []
    Resultado: CADENA RECHAZADA
    Tampoco llego a ocurrir ningun bloqueo, se procesan todos los simbolos siendo la ultima la 'b' que lleva a un estado destino activo [5]
    y su conjunto expandido [1, 2, 4, 5, 6, 7], pero ya que este conjunto no incluye al estado de aceptación 10 del AFN, la cadena no puede ser aceptada.

    '''
    '''
    Prueba del codigo anterior
    Evaluando la cadena: 'aabb'
    Estado Activo Inicial e-closure(0): [0, 1, 2, 4, 7]
    Paso 1 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
    Paso 2 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
    Paso 3 ('b'): mover -> [5, 9] | e-closure -> [1, 2, 4, 5, 6, 7, 9]
    Paso 4 ('b'): mover -> [5, 10] | e-closure -> [1, 2, 4, 5, 6, 7, 10]
    Estados Activos Finales: [1, 2, 4, 5, 6, 7, 10] | Intersección F: [10]
    Resultado: CADENA ACEPTADA

        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}

    La ejecución muestra al estado expandido de s0 como [0, 1, 2, 4, 7] debido a que primero el 0 se expande a incluir 1 y 7, y despues el estado 1 tambien se expande a incluir 2 y 4 mediante epsilon.
    Despues se intenta hacer el movimiento con a a travez de cada uno de estos [0, 1, 2, 4, 7], de estos solo existe las transiciones de 2-> {3}  y 7-> {8}. El conjunto de estados destino [3,8] pasa a ser expandido
    y se transforma en [1, 2, 3, 4, 6, 7, 8] ya que  las expansiones por transiciones vacias fueron asi 3->6 , 6-> 1 , 6-> 7, 1->2 , 1->4  y el propio 8 que no se expandio más alla de a si mismo.

    '''

=== EMULADOR DE AFN-e (Simulación Directa) ===

Evaluando la cadena: 'aa'
La pila comienza con el estado '0'. Pila = [0]
El estado '0' salio de la pila. Pila = []
El estado '1' entro en la pila. Pila = [1]
El estado '7' entro en la pila. Pila = [1, 7]
El estado '7' salio de la pila. Pila = [1]
El estado '1' salio de la pila. Pila = []
El estado '2' entro en la pila. Pila = [2]
El estado '4' entro en la pila. Pila = [2, 4]
El estado '4' salio de la pila. Pila = [2]
El estado '2' salio de la pila. Pila = []
La lista de estados visitados con ayuda de la pila es = {0, 1, 2, 4, 7}

En los estados expandidos {0, 1, 2, 4, 7} se uso la transición a-> y se pudo llegar a {8, 3}
Lista de estados destino a expandir: {8, 3}###############

La pila comienza con el estado '8'. Pila = [8]
El estado '8' salio de la pila. Pila = []
La lista de estados visitados con ayuda de la pila es = {8}

La pila comienza con el estado '3'. Pila = [3]
El estado '3' salio de la pila. Pila = []
El estado '6' entro en l

### Desafío 4: Extensión de la Construcción de Thompson (Nivel: Intermedio)  

Script de referencia: erafnEpsilon.py  

**Objetivo:** Entender la programación orientada a objetos aplicada a estructuras de grafos y la aplicación práctica de reglas inductivas en compiladores.  

**Consigna para el estudiante:**  

>Estudio de arquitectura POO: Analice cómo la clase `State` gestiona las referencias a objetos vecinos mediante listas de adyacencia y cómo la clase `ThompsonNFA` encapsula los puntos de entrada (`start`) y salida (`accept`).  

>Desafío de implementación (Operador Opcional `?`): Añada un nuevo método estático a la clase `ThompsonNFA` llamado `build_optional(nfa: 'ThompsonNFA') -> 'ThompsonNFA'` que implemente la regla inductiva para la cerradura opcional ($r?$, es decir, cero o una repetición). Pista: requiere conectar el nuevo estado inicial con el estado inicial del sub-autómata y directamente con el nuevo estado final mediante épsilon, además de conectar el final anterior al nuevo final.  

>Prueba de integración: Utilice su nuevo método para construir el AFN correspondiente a la expresión regular opcional `a?` y ejecute la impresión de la lista de adyacencia para verificar sus conexiones.

In [ ]:
# ==============================================================================
# Tema: Algoritmo de Construcción de Thompson
# ==============================================================================

from typing import Set, Dict, Tuple, List

class State: #Estructura para representar a cada estado de manera individual, cuenta con un diccionario donde se registran su abecedario y transiciones
    """Representa un estado individual de un autómata Thompson."""
    _id_counter = 0

    def __init__(self):
        self.id = State._id_counter
        State._id_counter += 1
        # Diccionario de transiciones: símbolo -> lista de estados destino
        self.transitions: Dict[str, List['State']] = {} # ejemplo del diccionario:  [ Simbolo 'a' : {lista de estados} ]

    def add_transition(self, symbol: str, target_state: 'State'): # Simbolo y estado destino
        if symbol not in self.transitions: #Si es un nuevo simbolo, pasa a incluirse en el diccionario. Como tal no hay una lista de abecedario global, sino que cada estado tendra sus simbolos registrados
            self.transitions[symbol] = []
        self.transitions[symbol].append(target_state) #Agrega el nuevo estado a la lista de estados vinculada al simbolo.

    def __repr__(self):
        return f"q{self.id}"

class ThompsonNFA:
    """Encapsula un sub-autómata Thompson con exactamente UN inicio y UN fin."""
    EPSILON = 'ε' #Se declara a el simbolo epsilon como el caracter ε

    def __init__(self, start: State, accept: State):
        self.start = start
        self.accept = accept #Solo un estado de aceptación

    @staticmethod
    def build_symbol(symbol: str) -> 'ThompsonNFA': #Genera mini automatas simples de un solo simbolo. Representa a las expresiones simples como "a" = O--a-->O
        """Regla Base: Construcción para un símbolo a in Sigma (o epsilon)."""
        start = State() #Estado de inicio de este automata especifico
        accept = State() #Estado fin de este automata especifico
        start.add_transition(symbol, accept) #Se crea la transición del nodo de inicio al destino usando el simbolo
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_union(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA': #Genera un automata más complejo compuesto de otros 2 automatas,  dos nodos auxiliares y 4 transiciones vacias.
    # Representa la estructura de union '|' que es una bifurcación
        """Regla Inductiva 1: Unión (nfa1 | nfa2)."""
        start = State() #Nuevo nodo auxiliar (inicio)
        accept = State() #Nuevo nodo auxiliar (destino)

        # Transiciones epsilon desde nuevo inicio a inicios de nfa1 y nfa2
        start.add_transition(ThompsonNFA.EPSILON, nfa1.start)
        start.add_transition(ThompsonNFA.EPSILON, nfa2.start)

        # Transiciones epsilon desde finales de nfa1 y nfa2 a nuevo final
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, accept)
        nfa2.accept.add_transition(ThompsonNFA.EPSILON, accept)
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_concat(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA': #Genera un automata un poco más complejo compuesto de otros 2 automatas conectados mediante una transición vacia.
        """Regla Inductiva 2: Concatenación (nfa1 . nfa2)."""
        # Conectar final de nfa1 con inicio de nfa2 mediante epsilon
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        return ThompsonNFA(nfa1.start, nfa2.accept)

    @staticmethod
    def build_kleene(nfa: 'ThompsonNFA') -> 'ThompsonNFA': #Automata de estructura compleja compuesta de 1 automata, 2 nodos auxiliares y 4 transiciones vacias.
    # Es la estructura de clausura de kleene '*' la cual permite el procesamiento de 0 o n veces un simbolo mediante el uso de transiciones para crear bucles
        """Regla Inductiva 3: Clausura de Kleene (nfa*)."""
        start = State()
        accept = State()

        # 1. Transición epsilon de bypass (0 repeticiones)
        start.add_transition(ThompsonNFA.EPSILON, accept)
        # 2. Transición epsilon hacia inicio del sub-autómata
        start.add_transition(ThompsonNFA.EPSILON, nfa.start)
        # 3. Transición epsilon desde final del sub-autómata hacia nuevo final
        nfa.accept.add_transition(ThompsonNFA.EPSILON, accept)
        # 4. Transición epsilon de retroceso (loop de repetición)
        nfa.accept.add_transition(ThompsonNFA.EPSILON, nfa.start)
        return ThompsonNFA(start, accept)

    def print_nfa(self): #Recorre el automata y lo imprime
        """Imprime la lista de adyacencia de todas las transiciones del autómata."""
        visited = set()
        stack = [self.start] #Estructura usada para navegar los distintos caminos del automata
        print(f"\n=== AFN Thompson (Inicio: {self.start}, Aceptación: {self.accept}) ===")

        while stack:
            state = stack.pop()
            if state.id in visited:
                continue #Si es un estado que ya a sido visitado, el ciclo pasa directamente a su siguiente iteración
            visited.add(state.id)

            for symbol, targets in state.transitions.items(): # Las transiciones se guardan en un diccionaario, .items esta devolviendo algo asi: 'ε': [q1, q3]  o 'a': [q2]
            #Los simbolos son una capa superior en el bucle, un simbolo puede conllevar varios estados destino
                for target in targets: #Recorre estado por estado
                    print(f"  {state} --({symbol})--> {target}")
                    if target.id not in visited:
                        stack.append(target) #Se encargara de poder explorar los caminos que salen de los nodos que aun no habian sido visitados

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Reset contador de estados
    State._id_counter = 0

    # Construir inductivamente el AFN para la expresión: (a | b)*
    nfa_a = ThompsonNFA.build_symbol('a')      # a , automata simple
    nfa_b = ThompsonNFA.build_symbol('b')      # b , automata simple
    nfa_ab = ThompsonNFA.build_union(nfa_a, nfa_b)  # (a | b) , automata que une los dos anteriores
    nfa_star = ThompsonNFA.build_kleene(nfa_ab)     # (a | b)* , al anterior se le aplica la clausula de kleene

    nfa_star.print_nfa()

### Desafío 3: Construcción de Subconjuntos y Equivalencia de Autómatas (Nivel: Intermedio)  

Script de referencia: afnafd.py y AFDeterminista.py  

**Objetivo:** Conectar la teoría del paso de AFN a AFD mediante el algoritmo de construcción de subconjuntos, analizando la explosión o optimización de estados.  

**Consigna para el estudiante:**  

>Análisis de salida: Ejecute `afnafd.py` y examine la correspondencia entre los subconjuntos numéricos generados (p. ej., `{0, 1, 2, 4, 7}`) y las etiquetas alfabéticas asignadas (`A`, `B`, `C`, etc.).  

>Validación cruzada: Tome la matriz de transición resultante (`d_tran`) generada por el script de subconjuntos para el lenguaje $(a\vert{}b)^*abb$ y conviértala manualmente a una tabla tabular similar a la que imprime `AFDeterminista.py`.  

>Pregunta de reflexión técnica: En su reporte de práctica, responda: ¿Por qué el algoritmo utiliza una cola de índices no marcados (`unmarked_indices`) y qué pasaría con la ejecución si omitimos la verificación `if u_set in d_states`?

### Formato de Entrega / Portafolio de Evidencias
1. Trabajen en equipos de dos personas, entregando como reporte en el portafolio de evidencias:

*   El código fuente modificado con comentarios explicativos.

*   Capturas de pantalla o salidas de consola de las pruebas realizadas.

*   Respuesta conceptual a las preguntas de análisis lógico integradas en cada desafío.

2. Guarda este cuaderno con todas las salidas ejecutadas (`Archivo` -> `Guardar una copia en GitHub` / `PEREZ JUAN PSSB I Colab U3.ipynb`).

3. Registra en **Moodle** el enlace del cuaderno ejecutable en Colab y el de tu repositorio conteniendo la especificación formal del proyecto.